In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class OverAllState(TypedDict):
    initial_state: str
    parallel_node_a_1: str
    parallel_node_a_2: str
    node_b_output: str

def parallel_node_a_1(state: OverAllState) -> OverAllState:
    return {
        "parallel_node_a_1": "并行节点A-1的输出"
    }

def parallel_node_a_2(state: OverAllState) -> OverAllState:
    return {
        "parallel_node_a_2": "并行节点A-2的输出"
    }

def node_b(state: OverAllState) -> OverAllState:
    return {
        "node_b_output": "节点B的输出"
    }

builder = StateGraph(state_schema=OverAllState)
builder.add_node("parallel_node_a_1", parallel_node_a_1)
builder.add_node("parallel_node_a_2", parallel_node_a_2)
builder.add_node("node_b", node_b)
builder.add_edge(START, "parallel_node_a_1")
builder.add_edge(START, "parallel_node_a_2")
builder.add_edge(["parallel_node_a_1", "parallel_node_a_2"], "node_b")
builder.add_edge("node_b", END)

graph = builder.compile()

for chunk in graph.stream(
    {"initial_state": "初始状态"},
    stream_mode=["tasks"]
):
    print(chunk)

('tasks', {'id': '83c1d74e-416f-4eca-f28c-3312becc3dcc', 'name': 'parallel_node_a_1', 'input': {'initial_state': '初始状态'}, 'triggers': ('branch:to:parallel_node_a_1',)})
('tasks', {'id': 'ca20216a-88cd-7e39-ee20-60090c58ea2b', 'name': 'parallel_node_a_2', 'input': {'initial_state': '初始状态'}, 'triggers': ('branch:to:parallel_node_a_2',)})
('tasks', {'id': '83c1d74e-416f-4eca-f28c-3312becc3dcc', 'name': 'parallel_node_a_1', 'error': None, 'result': {'parallel_node_a_1': '并行节点A-1的输出'}, 'interrupts': []})
('tasks', {'id': 'ca20216a-88cd-7e39-ee20-60090c58ea2b', 'name': 'parallel_node_a_2', 'error': None, 'result': {'parallel_node_a_2': '并行节点A-2的输出'}, 'interrupts': []})
('tasks', {'id': '74777aec-73d2-fc4b-3259-52aa0a5c4542', 'name': 'node_b', 'input': {'initial_state': '初始状态', 'parallel_node_a_1': '并行节点A-1的输出', 'parallel_node_a_2': '并行节点A-2的输出'}, 'triggers': ('branch:to:node_b', 'join:parallel_node_a_1+parallel_node_a_2:node_b')})
('tasks', {'id': '74777aec-73d2-fc4b-3259-52aa0a5c4542', 'nam